<a href="https://colab.research.google.com/github/Leonanda1013/DataLoverz/blob/main/Competition/GammaFest_IPB/DataCleaning_DSC_gamma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
html = """

<style>
  .wrap { padding: 1rem 0; font-size: 14px; }
  table { width: 100%; border-collapse: collapse; }
  th { font-weight: 500; font-size: 13px; text-align: left; padding: 8px 10px; border-bottom: 1.5px solid var(--color-border-primary); color: var(--color-text-secondary); }
  td { padding: 7px 10px; border-bottom: 1px solid var(--color-border-tertiary); vertical-align: top; font-size: 13px; color: var(--color-text-primary); }
  tr:hover td { background: var(--color-background-secondary); }
  .badge { display: inline-block; padding: 2px 8px; border-radius: 999px; font-size: 11px; font-weight: 500; }
  .keep   { background: #EAF3DE; color: #27500A; }
  .drop   { background: #FCEBEB; color: #791F1F; }
  .maybe  { background: #FAEEDA; color: #633806; }
  .section { background: var(--color-background-secondary); font-weight: 500; color: var(--color-text-secondary); font-size: 12px; letter-spacing: .03em; }
  .missing { font-size: 11px; color: var(--color-text-tertiary); }
</style>
<div class="wrap">
  <table>
    <thead>
      <tr>
        <th style="width:32%">Fitur</th>
        <th style="width:12%">Status</th>
        <th style="width:12%">Missing</th>
        <th>Alasan</th>
      </tr>
    </thead>
    <tbody>
      <tr><td colspan="4" class="section">— Identitas & Admin (Hapus semua)</td></tr>
      <tr><td>Id</td><td><span class="badge drop">Hapus</span></td><td class="missing">0%</td><td>Identifier unik, tidak mengandung informasi prediktif</td></tr>
      <tr><td>match_id</td><td><span class="badge drop">Hapus</span></td><td class="missing">0%</td><td>Hanya penanda pertandingan, bukan fitur</td></tr>
      <tr><td>date</td><td><span class="badge maybe">Olah dulu</span></td><td class="missing">0%</td><td>Bisa di-extract: tahun, bulan, atau era untuk trend temporal</td></tr>

      <tr><td colspan="4" class="section">— Kondisi Pertandingan</td></tr>
      <tr><td>gender</td><td><span class="badge keep">Pakai</span></td><td class="missing">0%</td><td>Rata-rata gol berbeda antara kompetisi pria & wanita</td></tr>
      <tr><td>is_home</td><td><span class="badge keep">Pakai</span></td><td class="missing">0%</td><td>Home advantage signifikan dalam sepak bola</td></tr>
      <tr><td>neutral</td><td><span class="badge keep">Pakai</span></td><td class="missing">0%</td><td>Mengurangi home advantage, penting untuk model</td></tr>
      <tr><td>tournament</td><td><span class="badge keep">Pakai</span></td><td class="missing">0%</td><td>Intensitas laga berbeda (World Cup vs Friendly) → gol berbeda</td></tr>
      <tr><td>venue_country</td><td><span class="badge drop">Hapus</span></td><td class="missing">0%</td><td>Sudah ter-capture oleh is_home + neutral + altitude + temperature</td></tr>
      <tr><td>team / opponent</td><td><span class="badge maybe">Opsional</span></td><td class="missing">0%</td><td>Bisa dipakai jika encode sebagai ELO/rank; raw string → high cardinality</td></tr>
      <tr><td>confederation_team/opp</td><td><span class="badge keep">Pakai</span></td><td class="missing">0%</td><td>Gaya bermain & rata-rata gol berbeda antar konfederasi</td></tr>

      <tr><td colspan="4" class="section">— Performa Tim (Fitur utama)</td></tr>
      <tr><td>elo_team / elo_opponent</td><td><span class="badge keep">Pakai</span></td><td class="missing">0%</td><td>Kekuatan relatif tim — sinyal prediktif terkuat, missing 0%</td></tr>
      <tr><td>team_points_last5</td><td><span class="badge keep">Pakai</span></td><td class="missing">~0.4%</td><td>Form terkini tim — sangat relevan</td></tr>
      <tr><td>opp_points_last5</td><td><span class="badge keep">Pakai</span></td><td class="missing">~0.2%</td><td>Form terkini lawan</td></tr>
      <tr><td>points_last5_diff</td><td><span class="badge drop">Hapus</span></td><td class="missing">~0.6%</td><td>Derivasi dari dua kolom di atas → multikolinearitas</td></tr>
      <tr><td>team_gd_last5</td><td><span class="badge keep">Pakai</span></td><td class="missing">~0.4%</td><td>Selisih gol mencerminkan kekuatan attack & defense</td></tr>
      <tr><td>opp_gd_last5</td><td><span class="badge keep">Pakai</span></td><td class="missing">~0.2%</td><td>Selisih gol lawan</td></tr>
      <tr><td>gd_last5_diff</td><td><span class="badge drop">Hapus</span></td><td class="missing">~0.6%</td><td>Derivasi langsung → redundan</td></tr>
      <tr><td>h2h_points_last5</td><td><span class="badge keep">Pakai</span></td><td class="missing">~15%</td><td>Head-to-head penting; impute 0 untuk matchup baru</td></tr>
      <tr><td>h2h_gd_last5</td><td><span class="badge keep">Pakai</span></td><td class="missing">~15%</td><td>Head-to-head goal difference; impute 0</td></tr>
      <tr><td>days_since_last_match_team</td><td><span class="badge keep">Pakai</span></td><td class="missing">~0.4%</td><td>Fatigue & rotasi roster berpengaruh</td></tr>
      <tr><td>days_since_last_match_opp</td><td><span class="badge keep">Pakai</span></td><td class="missing">~0.2%</td><td>Fatigue lawan</td></tr>
      <tr><td>team_points_last10</td><td><span class="badge maybe">Opsional</span></td><td class="missing">~0.4%</td><td>Korelasi tinggi dengan last5; cek VIF sebelum masukkan keduanya</td></tr>
      <tr><td>opp_points_last10</td><td><span class="badge maybe">Opsional</span></td><td class="missing">~0.2%</td><td>Sama seperti di atas</td></tr>
      <tr><td>team_avg_goals_last5</td><td><span class="badge keep">Pakai</span></td><td class="missing">~0.4%</td><td>Proxy kekuatan serangan langsung ke target</td></tr>
      <tr><td>team_avg_conceded_last5</td><td><span class="badge keep">Pakai</span></td><td class="missing">~0.4%</td><td>Proxy kekuatan pertahanan</td></tr>
      <tr><td>opp_avg_goals_last5</td><td><span class="badge keep">Pakai</span></td><td class="missing">~0.2%</td><td>Proxy serangan lawan</td></tr>
      <tr><td>opp_avg_conceded_last5</td><td><span class="badge keep">Pakai</span></td><td class="missing">~0.2%</td><td>Proxy pertahanan lawan</td></tr>
      <tr><td>team_win_rate_last10</td><td><span class="badge keep">Pakai</span></td><td class="missing">~0.4%</td><td>Win rate long-term mencerminkan konsistensi</td></tr>
      <tr><td>opp_win_rate_last10</td><td><span class="badge keep">Pakai</span></td><td class="missing">~0.2%</td><td>Konsistensi lawan</td></tr>
      <tr><td>rank_team / rank_opponent</td><td><span class="badge maybe">Opsional</span></td><td class="missing">~53%</td><td>Missing sangat banyak; ELO sudah cover info ini — pertimbangkan drop</td></tr>
      <tr><td>rank_diff</td><td><span class="badge drop">Hapus</span></td><td class="missing">~56%</td><td>Derivasi + missing >50% — tidak worth impute</td></tr>
      <tr><td>rank_missing_team/opp</td><td><span class="badge keep">Pakai</span></td><td class="missing">0%</td><td>Flag missingness → bisa jadi sinyal era/era kompetisi</td></tr>

      <tr><td colspan="4" class="section">— Geografis & Sosio-Ekonomi</td></tr>
      <tr><td>altitude_venue</td><td><span class="badge keep">Pakai</span></td><td class="missing">~26%</td><td>Ketinggian tinggi terbukti pengaruhi performa fisik</td></tr>
      <tr><td>temperature_venue</td><td><span class="badge keep">Pakai</span></td><td class="missing">~22%</td><td>Suhu ekstrem mengurangi intensitas, imputasi median aman</td></tr>
      <tr><td>distance_travel_team/opp</td><td><span class="badge keep">Pakai</span></td><td class="missing">~39%</td><td>Fatigue perjalanan berpengaruh; impute + tambah flag</td></tr>
      <tr><td>population_team/opp</td><td><span class="badge maybe">Opsional</span></td><td class="missing">~19%</td><td>Sinyal lemah; pool talent kasar — uji feature importance dulu</td></tr>
      <tr><td>gdp_per_capita_team/opp</td><td><span class="badge maybe">Opsional</span></td><td class="missing">~33%</td><td>Proxy infrastruktur sepak bola; sinyal lemah, missing banyak</td></tr>
    </tbody>
  </table>
</div>

"""

In [ ]:
# ── Semua import di sini ──────────────────────────────────────
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings("ignore")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from IPython.display import display, HTML

In [ ]:
display(HTML(html))

In [ ]:
"""
Preprocessing Pipeline — Football Match Goal Prediction
========================================================
Output : train_clean.csv  (siap pakai, sudah include target)
         test_clean.csv   (siap pakai, tanpa target)

Cara pakai temanmu:
    train = pd.read_csv("train_clean.csv")
    test  = pd.read_csv("test_clean.csv")
    X = train.drop(columns=["team_goals", "opp_goals"])
    y = train[["team_goals", "opp_goals"]]
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings("ignore")


In [ ]:


# ═══════════════════════════════════════════════════════════════
# KONFIGURASI
# ═══════════════════════════════════════════════════════════════
TARGET_COLS = ["team_goals", "opp_goals"]

COLS_DROP_TRAIN = [
    "Id", "match_id", "venue_country", "team", "opponent",
    "team_points_last5", "opp_points_last5", "points_last5_diff",
    "team_gd_last5", "opp_gd_last5", "gd_last5_diff",
    "h2h_points_last5", "h2h_gd_last5",
    "days_since_last_match_team", "days_since_last_match_opp",
    "team_points_last10", "opp_points_last10",
    "team_avg_goals_last5", "team_avg_conceded_last5",
    "opp_avg_goals_last5", "opp_avg_conceded_last5",
    "team_win_rate_last10", "opp_win_rate_last10",
    "elo_team", "elo_opponent",
    "rank_team", "rank_opponent", "rank_diff",
    "rank_missing_team", "rank_missing_opp",
]

COLS_DROP_TEST = [
    "Id", "match_id", "venue_country", "team", "opponent",
]

COLS_IMPUTE_MEDIAN = [
    "population_team", "population_opp",
    "gdp_per_capita_team", "gdp_per_capita_opp",
    "altitude_venue",
    "distance_travel_team", "distance_travel_opp",
    "temperature_venue",
]

COLS_LABEL_ENCODE = ["gender"]
COLS_OHE          = ["confederation_team", "confederation_opp"]
COLS_FREQ_ENCODE  = ["tournament"]

COLS_NO_SCALE = {
    "is_home", "neutral", "gender",
    "year", "month", "match_era",
    "team_goals", "opp_goals",
}

In [ ]:


# ═══════════════════════════════════════════════════════════════
# LOAD
# ═══════════════════════════════════════════════════════════════
def load_data(train_path: str, test_path: str = None):
    train = pd.read_csv(train_path)
    test  = pd.read_csv(test_path) if test_path else None
    print(f"Train shape : {train.shape}")
    if test is not None:
        print(f"Test  shape : {test.shape}")
    return train, test


In [ ]:

# ═══════════════════════════════════════════════════════════════
# STEP 1 — DROP
# ═══════════════════════════════════════════════════════════════
def drop_columns(df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    df   = df.copy()
    cols = COLS_DROP_TRAIN if is_train else COLS_DROP_TEST
    drop = [c for c in cols if c in df.columns]
    df   = df.drop(columns=drop)
    print(f"[Step 1] Drop {len(drop)} kolom → shape: {df.shape}")
    return df


In [ ]:


# ═══════════════════════════════════════════════════════════════
# STEP 2 — DATE FE
# ═══════════════════════════════════════════════════════════════
def get_era(year: int) -> int:
    if year < 1950:   return 0
    elif year < 1970: return 1
    elif year < 1990: return 2
    elif year < 2000: return 3
    elif year < 2010: return 4
    elif year < 2018: return 5
    else:             return 6




def engineer_date_features(df: pd.DataFrame) -> pd.DataFrame:
    df          = df.copy()
    df["date"]  = pd.to_datetime(df["date"])
    df["year"]  = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["match_era"] = df["year"].apply(get_era)
    df          = df.drop(columns=["date"])
    print(f"[Step 2] Date FE → tambah: year, month, match_era")
    return df




In [ ]:

# ═══════════════════════════════════════════════════════════════
# STEP 3 — MISSING FLAGS
# ═══════════════════════════════════════════════════════════════
def create_missing_flags(df: pd.DataFrame) -> pd.DataFrame:
    df      = df.copy()
    created = []
    for col in COLS_IMPUTE_MEDIAN:
        if col in df.columns:
            df[f"{col}_missing"] = df[col].isna().astype(int)
            created.append(col)
    print(f"[Step 3] Buat {len(created)} flag missing")
    return df


In [ ]:

# ═══════════════════════════════════════════════════════════════
# STEP 4 — IMPUTASI
# ═══════════════════════════════════════════════════════════════
class MedianImputer:
    def __init__(self):
        self.medians = {}

    def fit(self, df: pd.DataFrame) -> "MedianImputer":
        for col in COLS_IMPUTE_MEDIAN:
            if col in df.columns:
                self.medians[col] = df[col].median()
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        for col, val in self.medians.items():
            if col in df.columns:
                df[col] = df[col].fillna(val)
        return df

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        return self.fit(df).transform(df)


def impute_missing(df: pd.DataFrame, imputer: MedianImputer = None, is_train: bool = True):
    if is_train:
        imputer = MedianImputer()
        df      = imputer.fit_transform(df)
    else:
        df = imputer.transform(df)
    print(f"[Step 4] Imputasi selesai → sisa missing: {df.isnull().sum().sum()}")
    return df, imputer


In [ ]:

# ═══════════════════════════════════════════════════════════════
# STEP 5 — ENCODING
# ═══════════════════════════════════════════════════════════════
class CategoricalEncoder:
    def __init__(self):
        self.label_encoders = {}
        self.ohe_categories = {}
        self.freq_maps      = {}

    def fit(self, df: pd.DataFrame) -> "CategoricalEncoder":
        for col in COLS_LABEL_ENCODE:
            if col in df.columns:
                le = LabelEncoder()
                le.fit(df[col].astype(str))
                self.label_encoders[col] = le

        for col in COLS_OHE:
            if col in df.columns:
                self.ohe_categories[col] = sorted(df[col].dropna().unique().tolist())

        for col in COLS_FREQ_ENCODE:
            if col in df.columns:
                self.freq_maps[col] = df[col].value_counts(normalize=True).to_dict()

        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        for col, le in self.label_encoders.items():
            if col in df.columns:
                df[col] = df[col].astype(str).apply(
                    lambda x: x if x in le.classes_ else le.classes_[0]
                )
                df[col] = le.transform(df[col])

        for col, cats in self.ohe_categories.items():
            if col in df.columns:
                for cat in cats:
                    df[f"{col}_{cat}"] = (df[col] == cat).astype(int)
                df = df.drop(columns=[col])

        for col, freq_map in self.freq_maps.items():
            if col in df.columns:
                df[col] = df[col].map(freq_map).fillna(0)

        return df

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        return self.fit(df).transform(df)


def encode_categoricals(df: pd.DataFrame, encoder: CategoricalEncoder = None, is_train: bool = True):
    if is_train:
        encoder = CategoricalEncoder()
        df      = encoder.fit_transform(df)
    else:
        df = encoder.transform(df)
    print(f"[Step 5] Encoding selesai → shape: {df.shape}")
    return df, encoder


In [ ]:

# ═══════════════════════════════════════════════════════════════
# STEP 6 — SCALING
# ═══════════════════════════════════════════════════════════════
class FeatureScaler:
    def __init__(self):
        self.scaler = StandardScaler()
        self.cols   = []

    def _get_cols(self, df: pd.DataFrame) -> list:
        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        return [
            c for c in num_cols
            if c not in COLS_NO_SCALE
            and not c.endswith("_missing")
            and df[c].nunique() > 2
        ]

    def fit(self, df: pd.DataFrame) -> "FeatureScaler":
        self.cols = self._get_cols(df)
        self.scaler.fit(df[self.cols])
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df       = df.copy()
        existing = [c for c in self.cols if c in df.columns]
        df[existing] = self.scaler.transform(df[existing])
        return df

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        return self.fit(df).transform(df)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# PIPELINE UTAMA
# ═══════════════════════════════════════════════════════════════
def run_pipeline(
    train_path: str,
    test_path: str = None,
    out_train: str = "train_clean.csv",
    out_test: str  = "test_clean.csv",
):
    print("=" * 60)
    print("MEMULAI PREPROCESSING PIPELINE")
    print("=" * 60)

    train_df, test_df = load_data(train_path, test_path)

    # Simpan Id test sebelum di-drop → untuk jaminan urutan baris
    test_ids = test_df["Id"].reset_index(drop=True) if test_df is not None else None

    # Step 1 — Drop
    train_df = drop_columns(train_df, is_train=True)
    if test_df is not None:
        test_df = drop_columns(test_df, is_train=False)

    # Step 2 — Date FE
    train_df = engineer_date_features(train_df)
    if test_df is not None:
        test_df = engineer_date_features(test_df)

    # Step 3 — Missing flags
    train_df = create_missing_flags(train_df)
    if test_df is not None:
        test_df = create_missing_flags(test_df)

    # Step 4 — Imputasi (fit hanya di train)
    train_df, imputer = impute_missing(train_df, is_train=True)
    if test_df is not None:
        test_df, _ = impute_missing(test_df, imputer=imputer, is_train=False)

    # Step 5 — Encoding (fit hanya di train)
    train_df, encoder = encode_categoricals(train_df, is_train=True)
    if test_df is not None:
        test_df, _ = encode_categoricals(test_df, encoder=encoder, is_train=False)

    # Selaraskan kolom test dengan train
    if test_df is not None:
        train_feat_cols = [c for c in train_df.columns if c not in TARGET_COLS]
        for col in train_feat_cols:
            if col not in test_df.columns:
                test_df[col] = 0
        test_df = test_df[train_feat_cols]
        print(f"[Align] Kolom test diselaraskan → {test_df.shape[1]} fitur")

    # Step 6 — Scaling (fit hanya di train)
    scaler   = FeatureScaler()
    train_df = scaler.fit_transform(train_df)
    if test_df is not None:
        test_df = scaler.transform(test_df)
    print(f"[Step 6] Scaling selesai")

    # Kembalikan Id ke test, letakkan di kolom pertama
    if test_df is not None and test_ids is not None:
        test_df = test_df.reset_index(drop=True)
        test_df.insert(0, "Id", test_ids)
        assert len(test_df) == len(test_ids), "Jumlah baris berubah!"
        print(f"[Check] Id test dikembalikan — urutan baris terjamin")

    # ── Simpan ke CSV ──────────────────────────────────────────
    train_df.to_csv(out_train, index=False)
    print(f"\n[Saved] {out_train}  → {train_df.shape}  (include kolom target)")

    if test_df is not None:
        test_df.to_csv(out_test, index=False)
        print(f"[Saved] {out_test}   → {test_df.shape}  (kolom pertama = Id, urutan terjamin)")

    print("\n" + "=" * 60)
    print("SELESAI — file CSV siap dibagi ke temanmu")
    print("=" * 60)
    print(f"\nKolom di train_clean.csv  ({train_df.shape[1]} kolom):")
    for i, col in enumerate(train_df.columns, 1):
        mark = " ← TARGET" if col in TARGET_COLS else ""
        print(f"  {i:>2}. {col}{mark}")

    return train_df, test_df


In [ ]:

# ─────────────────────────────────────────────
# JALANKAN
# ─────────────────────────────────────────────
train_clean, test_clean = run_pipeline(
    train_path="/content/drive/MyDrive/DATASET/deescuy/train.csv",
    test_path="/content/drive/MyDrive/DATASET/deescuy/test.csv",       # hapus jika tidak ada
    out_train="train_clean.csv",
    out_test="test_clean.csv",
)


# ─────────────────────────────────────────────
# CARA PAKAI CLEAN (copy cell ini)
# ─────────────────────────────────────────────
# import pandas as pd
#
# train = pd.read_csv("train_clean.csv")
# test  = pd.read_csv("test_clean.csv")
#
# X = train.drop(columns=["team_goals", "opp_goals"])
# y = train[["team_goals", "opp_goals"]]
#
# # atau prediksi per target:
# y_team = train["team_goals"]
# y_opp  = train["opp_goals"]


MEMULAI PREPROCESSING PIPELINE
Train shape : (78772, 47)
Test  shape : (42422, 20)
[Step 1] Drop 30 kolom → shape: (78772, 17)
[Step 1] Drop 5 kolom → shape: (42422, 15)
[Step 2] Date FE → tambah: year, month, match_era
[Step 2] Date FE → tambah: year, month, match_era
[Step 3] Buat 8 flag missing
[Step 3] Buat 8 flag missing
[Step 4] Imputasi selesai → sisa missing: 0
[Step 4] Imputasi selesai → sisa missing: 0
[Step 5] Encoding selesai → shape: (78772, 39)
[Step 5] Encoding selesai → shape: (42422, 37)
[Align] Kolom test diselaraskan → 37 fitur
[Step 6] Scaling selesai
[Check] Id test dikembalikan — urutan baris terjamin

[Saved] train_clean.csv  → (78772, 39)  (include kolom target)
[Saved] test_clean.csv   → (42422, 38)  (kolom pertama = Id, urutan terjamin)

SELESAI — file CSV siap dibagi ke temanmu

Kolom di train_clean.csv  (39 kolom):
   1. gender
   2. is_home
   3. neutral
   4. tournament
   5. team_goals ← TARGET
   6. opp_goals ← TARGET
   7. population_team
   8. populati

In [ ]:
df_test = pd.read_csv('/content/test_clean.csv')

In [ ]:
df_testt

,Id,gender,is_home,neutral,tournament,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,...,confederation_team_OFC,confederation_team_UEFA,confederation_team_Unknown,confederation_opp_AFC,confederation_opp_CAF,confederation_opp_CONCACAF,confederation_opp_CONMEBOL,confederation_opp_OFC,confederation_opp_UEFA,confederation_opp_Unknown
0,M034984_Seychelles,0,1,0,-1.025993,-0.372509,-0.356751,0.354917,0.126695,-9.509898,...,0,0,0,0,1,0,0,0,0,0
1,M034984_Mauritius,0,0,0,-1.025993,-0.356751,-0.372509,0.126695,0.354917,-9.509898,...,0,0,0,0,1,0,0,0,0,0
2,M034985_Comoros,0,1,1,-1.025993,-0.365051,-0.368947,-0.464372,-0.018653,-9.509898,...,0,0,0,1,0,0,0,0,0,0
3,M034985_Maldives,0,0,1,-1.025993,-0.368947,-0.365051,-0.018653,-0.464372,-9.509898,...,0,0,0,0,1,0,0,0,0,0
4,M034986_Réunion,0,1,1,-1.025993,-0.243790,-0.086193,-0.359800,-0.534292,-9.509898,...,0,0,1,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42417,M049213_Ecuador,0,0,0,1.279461,-0.133040,-0.140611,-0.359800,-0.359800,-0.137277,...,0,0,0,0,0,0,0,0,1,0
42418,M049214_Norway,0,1,0,1.279461,-0.301297,-0.257337,-0.359800,-0.359800,-0.111034,...,0,1,0,0,0,0,0,0,1,0
42419,M049214_Switzerland,0,0,0,1.279461,-0.257337,-0.301297,-0.359800,-0.359800,-0.111034,...,0,1,0,0,0,0,0,0,1,0
42420,M049215_Kazakhstan,0,1,0,-1.036579,-0.114302,-0.362457,-0.359800,-0.359800,0.179518,...,0,1,0,0,1,0,0,0,0,0


In [ ]:
df_train = pd.read_csv('/content/train_clean.csv')

In [ ]:
df_trainn

,gender,is_home,neutral,tournament,team_goals,opp_goals,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,...,confederation_team_OFC,confederation_team_UEFA,confederation_team_Unknown,confederation_opp_AFC,confederation_opp_CAF,confederation_opp_CONCACAF,confederation_opp_CONMEBOL,confederation_opp_OFC,confederation_opp_UEFA,confederation_opp_Unknown
0,0,1,0,1.279461,0,0,-0.243790,-0.243790,-0.359800,-0.359800,...,0,1,0,0,0,0,0,0,1,0
1,0,0,0,1.279461,0,0,-0.243790,-0.243790,-0.359800,-0.359800,...,0,1,0,0,0,0,0,0,1,0
2,0,1,0,1.279461,4,2,-0.243790,-0.243790,-0.359800,-0.359800,...,0,1,0,0,0,0,0,0,1,0
3,0,0,0,1.279461,2,4,-0.243790,-0.243790,-0.359800,-0.359800,...,0,1,0,0,0,0,0,0,1,0
4,0,1,0,1.279461,2,1,-0.243790,-0.243790,-0.359800,-0.359800,...,0,1,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78767,0,0,1,-1.025993,1,1,-0.370929,-0.086193,-0.359800,-0.534292,...,0,0,1,0,1,0,0,0,0,0
78768,0,1,0,-1.025993,0,0,-0.372509,-0.365051,0.354917,-0.464372,...,0,0,0,0,1,0,0,0,0,0
78769,0,0,0,-1.025993,0,0,-0.365051,-0.372509,-0.464372,0.354917,...,0,0,0,0,1,0,0,0,0,0
78770,0,1,1,-1.025993,1,1,-0.368947,-0.356751,-0.018653,0.126695,...,0,0,0,0,1,0,0,0,0,0


In [ ]:
df_trainn.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78772 entries, 0 to 78771
Data columns (total 39 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   gender                        78772 non-null  int64  
 1   is_home                       78772 non-null  int64  
 2   neutral                       78772 non-null  int64  
 3   tournament                    78772 non-null  float64
 4   team_goals                    78772 non-null  int64  
 5   opp_goals                     78772 non-null  int64  
 6   population_team               78772 non-null  float64
 7   population_opp                78772 non-null  float64
 8   gdp_per_capita_team           78772 non-null  float64
 9   gdp_per_capita_opp            78772 non-null  float64
 10  altitude_venue                78772 non-null  float64
 11  distance_travel_team          78772 non-null  float64
 12  distance_travel_opp           78772 non-null  float64
 13  t